In [2]:
# Importing libraries and files
import pandas as pd
import numpy as np
import joblib
import time

from pathlib import Path
from catboost import CatBoostClassifier
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report
)

In [4]:
# Loading the data
DATA_DIR = Path("../data/processed")
MODEL_DIR = Path("../models")
REPORT_DIR = Path("../reports")

MODEL_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

X_train = pd.read_csv(DATA_DIR / "X_train.csv")
X_test = pd.read_csv(DATA_DIR / "X_test.csv")
y_train = pd.read_csv(DATA_DIR / "y_train.csv")["disease_encoded"]
y_test = pd.read_csv(DATA_DIR / "y_test.csv")["disease_encoded"]

print("Training:", X_train.shape)
print("Testing:", X_test.shape)


Training: (80000, 174)
Testing: (20000, 174)


In [5]:
# Training the model
cat_model = CatBoostClassifier(
    iterations=500,
    depth=8,
    learning_rate=0.1,
    loss_function="MultiClass",
    auto_class_weights="Balanced",
    random_seed=42,
    verbose=100,
    thread_count=-1
)

start_time = time.time()

cat_model.fit(X_train, y_train)

training_time = time.time() - start_time

print(f"Training time: {training_time:.2f} seconds")

0:	learn: 3.5809145	total: 822ms	remaining: 6m 50s
100:	learn: 0.9975519	total: 55.7s	remaining: 3m 39s
200:	learn: 0.6784674	total: 1m 48s	remaining: 2m 41s
300:	learn: 0.5301372	total: 2m 41s	remaining: 1m 46s
400:	learn: 0.4425822	total: 3m 34s	remaining: 52.9s
499:	learn: 0.3860293	total: 4m 27s	remaining: 0us
Training time: 267.96 seconds


In [7]:
# Evaluating the model
y_pred_cat = cat_model.predict(X_test).ravel().astype(int)

cat_metrics = {
    "Model": "CatBoost",
    "Accuracy": accuracy_score(y_test, y_pred_cat),
    "Macro F1": f1_score(
        y_test, y_pred_cat,
        average="macro",
        zero_division=0
    ),
    "Weighted F1": f1_score(
        y_test, y_pred_cat,
        average="weighted",
        zero_division=0
    ),
    "Training Time (seconds)": training_time
}

for name, value in cat_metrics.items():
    print(f"{name}: {value:.4f}" if isinstance(value, (float, np.floating)) else f"{name}: {value}")

Model: CatBoost
Accuracy: 0.5589
Macro F1: 0.3136
Weighted F1: 0.5679
Training Time (seconds): 267.9649


In [8]:
# Saving model and Results
joblib.dump(
    cat_model,
    MODEL_DIR / "catboost_model.joblib"
)

pd.DataFrame([cat_metrics]).to_csv(
    REPORT_DIR / "catboost_metrics.csv",
    index=False
)

print("CatBoost model and metrics saved successfully.")

CatBoost model and metrics saved successfully.
